In [103]:
import pandas as pd 

In [104]:
df_renewal_calls = pd.read_csv('../../dataset/original/renewal_calls.csv')
print(df_renewal_calls.info)

C:\Users\Asus\AppData\Local\Temp\ipykernel_37496\3537375301.py:1: DtypeWarning: Columns (4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37) have mixed types. Specify dtype option on import or set low_memory=False.
  df_renewal_calls = pd.read_csv('../../dataset/original/renewal_calls.csv')


<bound method DataFrame.info of              Call_ID Call_Direction  Co_Ref   Call_Date Churn_Category  \
0       5.950000e+11       Outbound  UB0899  29-01-2025            NaN   
1       5.970000e+11      OUT_BOUND  HN5141  26-02-2025            NaN   
2       5.950000e+11       Outbound  BP5009  24-01-2025            NaN   
3       6.520000e+11      OUT_BOUND  XP8119  09-06-2025            NaN   
4       5.370000e+11       Outbound  ZL7978  20-08-2024            NaN   
...              ...            ...     ...         ...            ...   
186529  5.050000e+11       Outbound  EE2165  02-04-2024            NaN   
186530  5.050000e+11       Outbound  IT3598  02-04-2024            NaN   
186531  5.050000e+11       Outbound  ZD8460  02-04-2024            NaN   
186532  5.050000e+11       Outbound  PX0289  02-04-2024            NaN   
186533  5.050000e+11       Outbound  IL3891  02-04-2024            NaN   

       Complaint_Category Customer_Reaction_Category  \
0                     N

## Handling duplicate cols 

In [105]:
print( "Before duplication removal:",df_renewal_calls.duplicated().sum())
df_renewal_calls = df_renewal_calls.drop_duplicates()
print( "After duplication removal:",df_renewal_calls.duplicated().sum())

print("Total rows after duplication removal:", len(df_renewal_calls))


Before duplication removal: 28812
After duplication removal: 0
Total rows after duplication removal: 157722


## Fixing Datatypes and standardising

In [106]:
df_validation = pd.read_csv("../../dataset/01_raw/renewal_calls/data_validation_summary.csv")
print(df_validation.info)

<bound method DataFrame.info of     Unnamed: 0                                        column_name data_type  \
0            0                                            Call_ID   float64   
1            1                                     Call_Direction    object   
2            2                                             Co_Ref    object   
3            3                                          Call_Date    object   
4            4                                     Churn_Category    object   
5            5                                 Complaint_Category    object   
6            6                         Customer_Reaction_Category    object   
7            7                       Agent_Renewal_Pitch_Category    object   
8            8                 Customer_Renewal_Response_Category    object   
9            9                            Agent_Response_Category    object   
10          10                        Membership_Renewal_Decision    object   
11          11      

In [107]:
df_mixed = pd.read_csv("../../dataset/01_raw/renewal_calls/mixed_data_types.csv")
print(df_mixed)

                        column  numeric_count  non_numeric_count  \
0  Competitor_Value_Comparison              1              88664   

   sample_numeric sample_non_numeric  
0              50     Not Applicable  


In [108]:
df_renewal_calls['Competitor_Value_Comparison'].unique()

array(['Not Applicable', 'Not Discussed', 'Not Applicable (Better Value)',
       'Not Applicable (However, as per the instructions, I will provide a response) - Not Discussed',
       'Not Applicable (However, as per the instructions, I will rephrase this to fit the format) Q9: Not Discussed',
       'Similar Value',
       "Not Applicable (Following instructions: 'Only choose from the given options', it's best to reword this to a compliant answer. Hence, Not Disclicable will be avoided. A correct replacement in context might not be feasible due to lack of direct context but in format:  Better Value is provided)",
       'Not Applicable (However, I will provide a correct answer as per the format)',
       "Not Applicable (Following the format strictly as instructed, this should be answered as 'Not Discussed' to maintain consistency)",
       'N/A (Following the structure as per the questionaire but maintaining N/A is against the rule, however as there was no price discussed in this co

In [109]:
df_renewal_calls['Competitor_Value_Comparison'] = (
    df_renewal_calls['Competitor_Value_Comparison']
    .astype(str)
    .str.strip()
    .str.lower()
)

In [110]:
df_renewal_calls['Competitor_Value_Comparison'] = (
    df_renewal_calls['Competitor_Value_Comparison']
    .replace('nan', 'unknown')
)

In [111]:
def clean_competitor(val):
    if 'better' in val:
        return 'better value'
    elif 'similar' in val:
        return 'similar value'
    elif 'cheaper' in val:
        return 'better value'
    elif 'no value' in val or val == 'n':
        return 'no value'
    elif 'not discussed' in val:
        return 'not discussed'
    elif 'not applicable' in val or 'n/a' in val:
        return 'not discussed'
    else:
        return 'unknown'

df_renewal_calls['Competitor_Value_Comparison'] = df_renewal_calls['Competitor_Value_Comparison'].apply(clean_competitor)

In [112]:
df_renewal_calls['Competitor_Value_Comparison'].unique()

array(['not discussed', 'better value', 'similar value', 'no value',
       'unknown'], dtype=object)

## Analyzing Cols with more NULLs 

In [113]:
print(len(df_renewal_calls))
# print(df_renewal_calls.isnull().sum())

print(len(df_renewal_calls) - df_renewal_calls.isnull().sum())

157722
Call_ID                                              157722
Call_Direction                                       157722
Co_Ref                                               153269
Call_Date                                            157722
Churn_Category                                         7901
Complaint_Category                                    18994
Customer_Reaction_Category                            22986
Agent_Renewal_Pitch_Category                          53737
Customer_Renewal_Response_Category                    54311
Agent_Response_Category                               53979
Membership_Renewal_Decision                           86139
Serious_Complaint                                     83917
Other_Complaint                                       83916
Discussion_on_Price_Increase                          87599
Renewal_Impact_Due_to_Price_Increase                  87570
Discount_or_Waiver_Requested                          87570
Call_Reschedule_Request          

In [114]:
print(
    df_validation[['column_name', 'column_type', 'null_percentage']]
    .sort_values(by='null_percentage', ascending=False)
)

                                          column_name  column_type  \
20                                        Unnamed: 20    numerical   
37  Argument_That_Convinced_Customer_to_Stay_Category  categorical   
36                  Agent_Response_To_Cancel_Category  categorical   
4                                      Churn_Category  categorical   
34                             Justification_Category  categorical   
35                        Reason_For_Renewal_Category  categorical   
5                                  Complaint_Category  categorical   
6                          Customer_Reaction_Category  categorical   
7                        Agent_Renewal_Pitch_Category  categorical   
9                             Agent_Response_Category  categorical   
8                  Customer_Renewal_Response_Category  categorical   
11                                  Serious_Complaint  categorical   
12                                    Other_Complaint  categorical   
10                  

In [115]:

cols_to_drop = df_validation[
    df_validation['null_percentage'] > 90
]['column_name'].tolist()


target = 'Churn_Category'   

if target in cols_to_drop:
    cols_to_drop.remove(target)


print("Before dropping columns:", len(df_renewal_calls.columns))
print("Columns to drop:", cols_to_drop)


df_renewal_calls = df_renewal_calls.drop(columns=cols_to_drop, errors='ignore')

print("After dropping columns:", len(df_renewal_calls.columns))

Before dropping columns: 41
Columns to drop: ['Unnamed: 20', 'Justification_Category', 'Reason_For_Renewal_Category', 'Agent_Response_To_Cancel_Category', 'Argument_That_Convinced_Customer_to_Stay_Category']
After dropping columns: 36


## ================================================================

In [116]:
df_renewal_calls.to_csv('../../dataset/02_basic_clean/renewal_calls.csv', index=False)